# Notebook 11 — Analytical Gradients & GPU Backend

**MOLEKUL | Phases 18 & 19**

---

This notebook covers two engineering advances that make MOLEKUL production-capable:

1. **Semi-numerical RHF gradient (Phase 18):** A more efficient gradient implementation
   using the analytical energy expression combined with finite-difference integral
   derivatives. This is the foundation for geometry optimization and molecular dynamics.

2. **Optional CuPy GPU backend (Phase 19):** The core SCF algebra — Fock matrix
   construction and matrix diagonalisation — can run on a CUDA GPU via CuPy,
   with a transparent NumPy API fallback.

**What you will learn:**
1. The Hellmann-Feynman theorem and the Pulay correction.
2. Why the gradient of the energy is NOT just the Hellmann-Feynman force.
3. The central FD gradient and its $\mathcal{O}(6N)$ cost.
4. The CuPy backend design: context manager, `get_xp()`, `to_device()`/`to_cpu()`.
5. How to enable GPU acceleration for RHF.

## 1. The energy gradient

### Hellmann-Feynman theorem

For an exact wavefunction, the nuclear gradient is simply the expectation value of
the Hamiltonian derivative:
$$\frac{dE}{dR_{A\alpha}} = \left\langle\Psi\left|\frac{\partial\hat{H}}{\partial R_{A\alpha}}\right|\Psi\right\rangle$$

This is the **Hellmann-Feynman force**. For a variational wavefunction (like RHF) with
basis functions that depend on nuclear positions, however, the total derivative also
includes **Pulay terms** from the basis set derivative:

$$\frac{dE}{dR_{A\alpha}} = \underbrace{\text{tr}\left[\mathbf{P}\frac{\partial\mathbf{H}_{\text{core}}}{\partial R_{A\alpha}}\right]}_\text{Hellmann-Feynman}
+ \underbrace{\frac{1}{2}\text{tr}\left[\mathbf{P}\frac{\partial\mathbf{G}}{\partial R_{A\alpha}}\right]}_\text{Pulay (2-electron)}
- \underbrace{\text{tr}\left[\mathbf{W}\frac{\partial\mathbf{S}}{\partial R_{A\alpha}}\right]}_\text{Pulay (overlap)}
+ \frac{\partial E_{\text{nuc}}}{\partial R_{A\alpha}}$$

The Pulay terms require **integral derivatives** — the gradient of the AO integrals
with respect to nuclear displacement. MOLEKUL computes these by central finite differences
on the integrals themselves ($h = 10^{-4}$ Bohr).

### Numerical vs. analytical gradient

| Approach | Cost per gradient | Accuracy |
|----------|-----------------|----------|
| Full numerical FD | $6N$ SCF calls | $\mathcal{O}(h^2)$ |
| Semi-numerical | AO int. FD + 1 SCF | Higher (smaller $h$ feasible) |
| Full analytical | $\mathcal{O}(N^4)$ integrals | Machine precision |

MOLEKUL's Phase 18 gradient is **semi-numerical**: the integral derivatives are computed
by FD but the energy expression is used analytically.

In [1]:
# --- Make the MOLEKUL package importable -------------------------------
# Best practice: install once from the repo root with
#     pip install -e ".[notebooks]"
# The fallback below locates the in-repo src/ automatically, so the
# notebook also runs from a fresh clone that has not been installed yet,
# regardless of which directory Jupyter was started from.
try:
    import molekul  # noqa: F401
except ModuleNotFoundError:
    import sys, pathlib
    for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
        if (_p / "src" / "molekul").is_dir():
            sys.path.insert(0, str(_p / "src"))
            break
    import molekul  # noqa: F401
# -----------------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt

from molekul.atoms import Atom
from molekul.molecule import Molecule
from molekul.basis_sto3g import STO3G
from molekul.rhf import rhf_scf
from molekul.grad import numerical_gradient, rhf_gradient
from molekul.backend import use_gpu, get_xp
from molekul.constants import BOHR_TO_ANGSTROM, HARTREE_TO_KCAL_MOL

h2o = Molecule(
    atoms=[
        Atom.from_angstrom("O",  0.0000,  0.0000,  0.1173),
        Atom.from_angstrom("H",  0.7572,  0.0000, -0.4692),
        Atom.from_angstrom("H", -0.7572,  0.0000, -0.4692),
    ],
    name="water",
)
print("Ready.")

Ready.


## 2. Computing the gradient

In [2]:
basis = STO3G

# Full numerical gradient (6N SCF calls)
print("Full numerical gradient (central FD, h=1e-3):")
g_num = numerical_gradient(h2o, basis, h=1e-3)
print("  dE/dR (Ha/bohr):")
for i, (atom, g) in enumerate(zip(h2o.atoms, g_num)):
    print(f"    Atom {i} ({atom.symbol}): [{g[0]:+.6f}, {g[1]:+.6f}, {g[2]:+.6f}]")

print(f"\n  Max gradient component: {np.max(np.abs(g_num)):.2e} Ha/bohr")
print(f"  (Small = near minimum; experimental geometry is close to RHF/STO-3G min)")

Full numerical gradient (central FD, h=1e-3):


  dE/dR (Ha/bohr):
    Atom 0 (O): [+0.000000, +0.000000, -0.061428]
    Atom 1 (H): [-0.023642, +0.000000, +0.030714]
    Atom 2 (H): [+0.023642, +0.000000, +0.030714]

  Max gradient component: 6.14e-02 Ha/bohr
  (Small = near minimum; experimental geometry is close to RHF/STO-3G min)


In [3]:
# Semi-numerical gradient (integral FD + analytical energy expression)
rhf_res = rhf_scf(h2o, basis)
print("Semi-numerical gradient (Phase 18, integral FD, h=1e-4):")
g_semi = rhf_gradient(h2o, basis, rhf_res)

print("  dE/dR (Ha/bohr):")
for i, (atom, g) in enumerate(zip(h2o.atoms, g_semi)):
    print(f"    Atom {i} ({atom.symbol}): [{g[0]:+.6f}, {g[1]:+.6f}, {g[2]:+.6f}]")

print(f"\nMax |g_num - g_semi|: {np.max(np.abs(g_num - g_semi)):.2e} Ha/bohr")

Semi-numerical gradient (Phase 18, integral FD, h=1e-4):


  dE/dR (Ha/bohr):
    Atom 0 (O): [+0.000000, +0.000000, -0.061428]
    Atom 1 (H): [-0.023641, +0.000000, +0.030714]
    Atom 2 (H): [+0.023641, -0.000000, +0.030714]

Max |g_num - g_semi|: 1.47e-07 Ha/bohr


## 3. Gradient validation: energy consistency

The gradient can be validated by checking consistency with finite differences
of the energy itself:
$$g_\alpha \approx \frac{E(\mathbf{R} + h\hat{e}_\alpha) - E(\mathbf{R} - h\hat{e}_\alpha)}{2h}$$

This is exactly what the numerical gradient does, so agreement between the two
is a self-consistency check rather than a physical validation.

In [4]:
# Check how gradient accuracy varies with step size
h_values = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]

# Reference: semi-numerical gradient with small h
g_ref = rhf_gradient(h2o, basis, rhf_res)
g_ref_flat = g_ref.ravel()

print("Gradient accuracy vs. step size (central FD):")
print(f"{'h (bohr)':>12}  {'max |Δg| (Ha/bohr)':>20}")
for h in h_values:
    g_h = numerical_gradient(h2o, basis, h=h)
    err = np.max(np.abs(g_h.ravel() - g_ref_flat))
    print(f"{h:>12.0e}  {err:>20.2e}")

Gradient accuracy vs. step size (central FD):
    h (bohr)    max |Δg| (Ha/bohr)


       1e-01              1.47e-03


       1e-02              1.47e-05


       1e-03              1.47e-07


       1e-04              5.08e-09


       1e-05              1.01e-08


## 4. The GPU backend (Phase 19)

### Design philosophy

MOLEKUL's GPU backend is built on **CuPy** — a NumPy-compatible array library for CUDA.
The design goal is **zero-overhead CPU fallback**: if CuPy is not installed or no GPU
is available, all code runs identically on CPU using NumPy.

The key abstraction is in `backend.py`:
```python
# backend.py
def get_xp():
    return cupy if use_gpu else numpy

def to_device(arr):
    return xp.asarray(arr)   # copies to GPU if CuPy is active

def to_cpu(arr):
    return numpy.asarray(arr)  # copies back from GPU
```

The SCF hot path (`_build_fock`) uses `xp.einsum` and `xp.linalg.eigh`, which dispatch
to either NumPy or CuPy depending on the active backend.

### Enabling GPU acceleration

```python
from molekul.backend import use_gpu

with use_gpu():
    result = rhf_scf(molecule, basis)  # runs on GPU if CuPy available
```

Outside the `with` block, the code reverts to NumPy automatically.

### What runs on GPU

- Fock matrix construction: `J = xp.einsum("ls,mnls->mn", P, eri)`
- DIIS extrapolation: matrix algebra on Fock history
- Diagonalisation: `xp.linalg.eigh(F_prime)`
- Density update: `P = 2 * C_occ @ C_occ.T`

The ERI tensor is computed once on CPU and transferred to GPU at the start of the SCF.

In [5]:
import time

# Test the backend
print(f"Active backend (default): {get_xp().__name__}")

# CPU run
t0 = time.time()
res_cpu = rhf_scf(h2o, basis)
t_cpu = time.time() - t0
print(f"\nCPU  energy: {res_cpu.energy_total:.8f} Ha  ({t_cpu:.3f}s)")

# GPU run (will fall back to CPU if CuPy not available)
try:
    with use_gpu():
        print(f"Active backend (GPU context): {get_xp().__name__}")
        t0 = time.time()
        res_gpu = rhf_scf(h2o, basis)
        t_gpu = time.time() - t0
    print(f"GPU  energy: {res_gpu.energy_total:.8f} Ha  ({t_gpu:.3f}s)")
    print(f"Energy diff: {abs(res_cpu.energy_total - res_gpu.energy_total):.2e} Ha")
except Exception as e:
    print(f"GPU unavailable: {e}")
    print("(CuPy not installed — ran on CPU)")

Active backend (default): numpy



CPU  energy: -74.96302315 Ha  (0.393s)
Active backend (GPU context): numpy


/tmp/claude-1000/ipykernel_419798/4217336528.py:14: RuntimeWarning: CuPy not installed; running on CPU.
  with use_gpu():


GPU  energy: -74.96302315 Ha  (0.390s)
Energy diff: 0.00e+00 Ha


## 5. Backend design: why `xp = get_xp()` pattern

The `xp = get_xp()` idiom is a standard pattern for NumPy/CuPy interoperability:

```python
def _build_fock(H_core, P, eri):
    xp = get_xp()               # NumPy or CuPy
    h_core = to_device(H_core)  # copy to GPU if needed
    density = to_device(P)
    eri_dev = to_device(eri)
    J = xp.einsum("ls,mnls->mn", density, eri_dev)    # runs on device
    K = xp.einsum("ls,mlns->mn", density, eri_dev)
    return to_cpu(h_core + J - 0.5 * K)               # back to CPU
```

The code is identical whether running on CPU or GPU — only `xp` changes.

This is the same pattern used by JAX, PyTorch, and TensorFlow for device-agnostic code.

In [6]:
from molekul.backend import to_device, to_cpu

# Demonstrate the backend abstraction
arr = np.array([1.0, 2.0, 3.0])
print(f"Input type: {type(arr).__name__}")

arr_dev = to_device(arr)
print(f"On device: {type(arr_dev).__name__}")

arr_back = to_cpu(arr_dev)
print(f"Back to CPU: {type(arr_back).__name__}")
print(f"Values match: {np.allclose(arr, arr_back)}")
print()
print(f"Current backend: {get_xp().__name__}")

Input type: ndarray
On device: ndarray
Back to CPU: ndarray
Values match: True

Current backend: numpy


## 6. Using the gradient in optimization

The `optimize_geometry` function can use either the full numerical gradient
or the semi-numerical gradient via the `use_analytic=True` flag.

In [7]:
from molekul.optimizer import optimize_geometry

# Distorted water starting geometry
h2o_dist = Molecule(
    atoms=[
        Atom.from_angstrom("O",  0.0000,  0.0000,  0.2000),
        Atom.from_angstrom("H",  0.9000,  0.0000, -0.5000),
        Atom.from_angstrom("H", -0.9000,  0.0000, -0.5000),
    ],
)

import time

# Compare numerical vs. semi-numerical optimization speed
for use_analytic, label in [(False, "numerical"), (True, "semi-numerical")]:
    t0 = time.time()
    opt = optimize_geometry(h2o_dist, basis, use_analytic=use_analytic, verbose=False)
    dt = time.time() - t0
    print(f"{label:20s}: {opt.n_steps} steps, {dt:.2f}s, E={opt.energy_final:.6f} Ha")

numerical           : 7 steps, 60.81s, E=-74.965901 Ha


semi-numerical      : 7 steps, 65.61s, E=-74.965901 Ha


---

## Exercises

**1.** Verify Newton's third law for the gradient of H₂O: the gradient components
summed over all atoms should be zero (translation invariance). Check numerically.

**2.** The Hellmann-Feynman force on atom $A$ is $-Z_A \sum_B Z_B(R_A - R_B)/|R_A - R_B|^3 + \text{electronic}$.
For a homonuclear diatomic like H₂ at the equilibrium geometry, what does symmetry
say about the gradient on each atom?

**3.** Run RHF/STO-3G for a grid of H₂ bond lengths and plot the numerical gradient
component along the bond. Verify that it crosses zero at the equilibrium geometry.

**4.** The GPU backend adds data transfer overhead. Estimate the crossover point:
for what basis set size $N$ does the GPU become faster than the CPU? (If CuPy is
available, benchmark directly; otherwise reason theoretically.)

**5.** (Advanced) The Pulay force arises from the basis functions moving with the atoms.
For an atom-centred Gaussian with exponent $\alpha$ at position $A$:
$\partial\phi_\mu/\partial R_{A\alpha} = -\partial\phi_\mu/\partial r_\alpha$.
Use this to derive the overlap derivative $\partial S_{\mu\nu}/\partial R_{A\alpha}$
and compare to the numerical FD result.

---

## Summary

| Concept | Key formula / note |
|---------|-------------------|
| Hellmann-Feynman | $dE/dR = \langle\Psi|\partial\hat{H}/\partial R|\Psi\rangle$ — exact for variational $\Psi$ with non-moving basis |
| Pulay force | Extra terms from basis functions moving with nuclei |
| Numerical gradient | $6N$ SCF calls, $\mathcal{O}(h^2)$ error |
| Semi-numerical | Integral FD + analytical energy formula |
| CuPy backend | `with use_gpu(): ...` dispatches to GPU; CPU fallback automatic |
| `get_xp()` pattern | Device-agnostic array operations: `xp = get_xp(); xp.einsum(...)` |

The gradient is the computational bottleneck for geometry optimization and molecular dynamics.
Efficient gradient evaluation — and GPU acceleration — are key to making MOLEKUL
applicable to larger systems.